<a href="https://colab.research.google.com/github/BrendoEnomoto/Deep-Learning-Projects/blob/main/classifica%C3%A7%C3%A3o/keras_fashion_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Descrição do Código

Este código implementa e compara dois modelos de aprendizado de máquina para classificar imagens de roupas da base de dados Fashion MNIST:

## Modelo 1: VGG-16

1. **Pré-processamento de dados:**
   - Carrega a base de dados Fashion MNIST.
   - Redimensiona as imagens para 32x32 pixels e as converte para RGB (3 canais de cor).
   - Normaliza os valores dos pixels para o intervalo [0, 1].
2. **Construção do modelo:**
   - Cria uma instância do modelo VGG-16 com pesos não pré-treinados.
   - Define o número de classes de saída como 10 (10 tipos de roupas).
   - Compila o modelo usando o otimizador Adam, função de perda `categorical_crossentropy` e métrica de precisão.
3. **Treinamento do modelo:**
   - Treina o modelo por 20 épocas com um tamanho de lote de 128.
   - Usa os dados de teste para validação durante o treinamento.

## Modelo 2: Modelo Personalizado

1. **Pré-processamento de dados:**
   - Carrega a base de dados Fashion MNIST.
   - Normaliza os valores dos pixels para o intervalo [0, 1].
2. **Construção do modelo:**
   - Define um modelo sequencial com camadas densas (fully connected).
   - Utiliza a função de ativação ReLU nas camadas ocultas e softmax na camada de saída.
   - Compila o modelo usando o otimizador Adam, função de perda `categorical_crossentropy` e métrica de precisão.
3. **Treinamento do modelo:**
   - Treina o modelo por 20 épocas com um tamanho de lote de 100.
   - Usa os dados de teste para validação durante o treinamento.


## Comparação:

O código define dois modelos, VGG-16 e um modelo personalizado, ambos treinados para classificar imagens de roupas da base de dados Fashion MNIST. Ele pré-processa os dados, configura cada modelo, treina cada modelo e avalia seu desempenho na tarefa de classificação de imagens. O modelo personalizado possui uma arquitetura mais simples em comparação com o modelo VGG-16, com quatro camadas densas. Ambos os modelos são compilados e treinados com configurações semelhantes (otimizador, função de perda, métrica), e o código acompanha a precisão do treinamento e da validação durante o processo de treinamento.

In [ ]:
import numpy as np
import keras
from keras import layers

# VGG-16

In [ ]:
# Model / data parameters
num_classes = 10 #numero de classes da camda de saida
input_shape = (28, 28, 1)#dimensão da imagem com expansão de dimensionalidade

# Load the data and split it between train and test sets
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Scale images to the [0, 1] range
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255

# Make sure images have shape (28, 28, 1)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

print("x_train shape:", x_train.shape)
print(x_train.shape[0], "train samples")
print(x_test.shape[0], "test samples")


# convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

x_train shape: (60000, 28, 28, 1)
60000 train samples
10000 test samples


In [ ]:
from tensorflow.image import resize
# Assuming x_train has shape (60000, 28, 28, 1)
x_train_resized = np.repeat(x_train, 3, -1)  # Repeat along the channel dimension
x_train_resized = resize(x_train_resized, (32, 32)) # Resize to (32, 32)

x_test_resized = np.repeat(x_test, 3, -1)  # Repeat along the channel dimension
x_test_resized = resize(x_test_resized, (32, 32)) # Resize to (32, 32)

print(x_train_resized.shape)
print(x_test_resized.shape)
# x_train_resized will now have shape (60000, 32, 32, 3)

(60000, 32, 32, 3)
(10000, 32, 32, 3)


In [ ]:
from keras.applications.vgg16 import VGG16
from keras.applications.vgg16 import preprocess_input

input_shape = (32, 32, 3)

vgg16_model = VGG16 (
    weights=None,
    input_shape=input_shape,
    classes=num_classes,
)

vgg16_model.summary()

vgg16_model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

Model: "vgg16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)           │ (None, 32, 32, 3)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block1_conv1 (Conv2D)                │ (None, 32, 32, 64)          │           1,792 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block1_conv2 (Conv2D)                │ (None, 32, 32, 64)          │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block1_pool (MaxPooling2D)           │ (None, 16, 16, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block2_conv1 (Conv2D)                │ (None, 16, 16, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block2_conv2 (Conv2D)                │ (None, 16, 16, 128)         │         147,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block2_pool (MaxPooling2D)           │ (None, 8, 8, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_conv1 (Conv2D)                │ (None, 8, 8, 256)           │         295,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_conv2 (Conv2D)                │ (None, 8, 8, 256)           │         590,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_conv3 (Conv2D)                │ (None, 8, 8, 256)           │         590,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block3_pool (MaxPooling2D)           │ (None, 4, 4, 256)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_conv1 (Conv2D)                │ (None, 4, 4, 512)           │       1,180,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_conv2 (Conv2D)                │ (None, 4, 4, 512)           │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_conv3 (Conv2D)                │ (None, 4, 4, 512)           │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block4_pool (MaxPooling2D)           │ (None, 2, 2, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_conv1 (Conv2D)                │ (None, 2, 2, 512)           │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_conv2 (Conv2D)                │ (None, 2, 2, 512)           │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_conv3 (Conv2D)                │ (None, 2, 2, 512)           │       2,359,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ block5_pool (MaxPooling2D)           │ (None, 1, 1, 512)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ fc1 (Dense)                          │ (None, 4096)                │       2,101,248 │
├──────────────────────────────────────┼─────────────────────────────┼──────────────

 Total params: 33,638,218 (128.32 MB)

 Trainable params: 33,638,218 (128.32 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
vgg16_model.fit(x_train_resized,
          y_train, batch_size=128,
          epochs=20,
          validation_data=(x_test_resized, y_test),
          verbose=1
)

Epoch 1/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 58s 108ms/step - accuracy: 0.0996 - loss: 2.3078 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 2/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 62s 70ms/step - accuracy: 0.0989 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 3/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 41s 70ms/step - accuracy: 0.0990 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 4/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 41s 70ms/step - accuracy: 0.1000 - loss: 2.3026 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 5/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 41s 70ms/step - accuracy: 0.0978 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 6/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 41s 70ms/step - accuracy: 0.0997 - loss: 2.3027 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 7/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 41s 70ms/step - accuracy: 0.1000 - loss: 2.3026 - val_accuracy: 0.1000 - val_loss: 2.3026
Epoch 8/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 33s 70ms/step - accuracy: 0.1004 - loss: 2.3026 -

# Modelo Personalizado

In [ ]:
# Model / data parameters
num_classes = 10 #numero de classes da camda de saida
input_shape = (28, 28, 1)#dimensão da imagem com expansão de dimensionalidade

# Load the data and split it between train and test sets
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Scale images to the [0, 1] range
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255

# Make sure images have shape (28, 28, 1)
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

print("x_train shape:", x_train.shape)
print(x_train.shape[0], "train samples")
print(x_test.shape[0], "test samples")


# convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

In [ ]:
model = keras.Sequential(
    [
        keras.Input(shape=input_shape),
        layers.Flatten(),
        layers.Dense(784, activation="relu", kernel_initializer='normal'),
        layers.Dense(1024, activation="relu", kernel_initializer='normal'),
        layers.Dense(2048, activation="relu", kernel_initializer='normal'),
        layers.Dense(2048, activation="relu", kernel_initializer='normal'),
        layers.Dense(num_classes, activation="softmax"),
    ]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ flatten (Flatten)                    │ (None, 784)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 784)                 │         615,440 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1024)                │         803,840 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 2048)                │       2,099,200 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 2048)                │       4,196,352 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 10)                  │          20,490 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 7,735,322 (29.51 MB)

 Trainable params: 7,735,322 (29.51 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
batch_size = 100
epochs = 20

model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_data=(x_test, y_test), verbose=1),

Epoch 1/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.3584 - loss: nan - val_accuracy: 0.3107 - val_loss: 8.5434
Epoch 2/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.2347 - loss: nan - val_accuracy: 0.4264 - val_loss: 8.3606
Epoch 3/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.4232 - loss: nan - val_accuracy: 0.4287 - val_loss: 8.3238
Epoch 4/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.4352 - loss: 8.3330 - val_accuracy: 0.4359 - val_loss: 8.2965
Epoch 5/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.4442 - loss: 8.2971 - val_accuracy: 0.4381 - val_loss: 8.2928
Epoch 6/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.4448 - loss: 8.2725 - val_accuracy: 0.4497 - val_loss: 8.2827
Epoch 7/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.4307 - loss: nan - val_accuracy: 0.4459 - val_loss: 8.2840
Epoch 8/20
600/600 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.4300 - loss: 8.2897 - val_accuracy: 0.4486 - val_l

KeyboardInterrupt: 